## 9. Useful things to know: EXISTS, HAVING, LIMIT, SELECT TOP, FETCH FIRST, CASE, WHEN, THEN, ELSE, END

In this notebook we will cover some more useful keywords and teach you to make Cases in SQL and some good practices inclusing how to close the connection when you are finished.

###### Keywords Covered - EXISTS, HAVING, LIMIT, SELECT TOP, FETCH FIRST, CASE, WHEN, THEN, ELSE, END

In [1]:
import warnings
warnings.filterwarnings("ignore", message="Pandas requires version")
import sqlite3
import pandas as pd

# Connect to database
data_conn = sqlite3.connect("pokemon.db")

data_conn.execute("DROP TABLE pokemon_items") # Delete to refresh any prior changes.

data_conn.execute("CREATE TABLE pokemon_items \
                        (name TEXT NOT NULL, \
                         effect TEXT NOT NULL)")

data_conn.execute("INSERT INTO pokemon_items (name, effect) VALUES \
                ('Poké Ball', 'Catches Pokémon.'), ('Great Ball', 'Catches Pokémon better.'), ('Ultra Ball', 'Catches Pokémon great!'), \
                ('Potion', 'Heals 20 HP.'), ('Super Potion', 'Heals 50 HP.'), ('Full Restore', 'Fully restores HP and cures all effects.'), \
                ('Revive', 'Revives your Pokémon.'), ('Escape Rope', 'Takes you to the entrance of a cave.')")

### EXISTS

The EXISTS operator is used in a WHERE clause to check whether a subquery returns any rows.

The EXISTS operator evaluates to TRUE if the subquery returns at least one row, and FALSE otherwise.

In [2]:
pd.read_sql("""SELECT type1, count(id)
FROM pokemon
WHERE EXISTS (
  SELECT id
  FROM pokemon
  WHERE type1 = 'water'
)
GROUP BY type1
LIMIT 5""", con=data_conn)

,type1,count(id)
0,bug,83
1,dark,45
2,dragon,37
3,electric,59
4,fairy,29


Below nothing is returend as no "bronze" type Pokemon exist.

This means EXISTS returns FALSE

In [3]:
# Using a type1 that does not exist in the table.
pd.read_sql("""SELECT type1, count(id)
FROM pokemon
WHERE EXISTS (
  SELECT id
  FROM pokemon
  WHERE type1 = 'bronze'
)
GROUP BY type1""", con=data_conn)

,type1,count(id)


By adding NOT before EXISTS, you reverse the FALSE to TRUE, so the EXIST clause reverts to TRUE.

In [4]:
pd.read_sql("""SELECT type1, count(id)
FROM pokemon
WHERE NOT EXISTS (
  SELECT id
  FROM pokemon
  WHERE type1 = 'bronze'
)
GROUP BY type1
LIMIT 3""", con=data_conn)

,type1,count(id)
0,bug,83
1,dark,45
2,dragon,37


### IF NOT EXISTS

Use: Combinantion of IF NOT and EXISTS that can be useful.

*Tip: It is good to create tables using CREATE TABLE IF NOT EXISTS to not throw an error.*

In [5]:
# Using CREATE TABLE IF NOT EXISTS.
data_conn.execute("CREATE TABLE IF NOT EXISTS pokemon_items \
                        (name TEXT NOT NULL, \
                         effect TEXT NOT NULL)")

In [6]:
# Not using IF NOT EXISTS throws an error.
data_conn.execute("CREATE TABLE pokemon_items \
                        (name TEXT NOT NULL, \
                         effect TEXT NOT NULL)")

OperationalError: table pokemon_items already exists

### HAVING

Use: Filtering GROUP BY results with aggregate functions such as COUNT and SUM.

In [7]:
pd.read_sql(sql="""SELECT type1, type2, COUNT(id) FROM pokemon
                   WHERE type1 = 'water'
                   GROUP BY type2 HAVING COUNT(id) > 3""", con=data_conn)

,type1,type2,COUNT(id)
0,water,NaN,74
1,water,dark,4
2,water,dragon,4
3,water,fairy,4
4,water,flying,7
5,water,ground,9
6,water,ice,4
7,water,psychic,6
8,water,rock,5


### LIMIT, SELECT TOP, FETCH FIRST

Use: Selecting the top X results of a query.

*Syntax: SELECT TOP X column_name FROM...*

*Syntax: SELECT... FETCH FIRST X ROWS ONLY*

In [8]:
pd.read_sql(sql="""SELECT name, id FROM pokemon
                   WHERE type1 = 'fire'
                   LIMIT 3""", con=data_conn)

,name,id
0,Charmander,4
1,Charmeleon,5
2,Charizard,6


### CASE WHEN THEN ELSE END

Use: Defining results based on certain conditions.

*Syntax:*

CASE

      WHEN condition1 THEN result1
  
      WHEN condition2 THEN result2
  
      WHEN conditionN THEN resultN
  
      ELSE default_result
  
END

*Tip: END AS column_name_to_be_duisplayed*

In [9]:
pd.read_sql(sql="""SELECT name, type1,
                    CASE
                      WHEN type1 = 'fire' THEN 'Blaine'
                      WHEN type1 = 'grass' THEN 'Erika'
                      WHEN type1 = 'water' THEN 'Misty'
                      ELSE 'Other Gym Leader'
                    END AS gymleader
                    FROM pokemon
                    LIMIT 15""", con=data_conn)

,name,type1,gymleader
0,Bulbasaur,grass,Erika
1,Ivysaur,grass,Erika
2,Venusaur,grass,Erika
3,Charmander,fire,Blaine
4,Charmeleon,fire,Blaine
5,Charizard,fire,Blaine
6,Squirtle,water,Misty
7,Wartortle,water,Misty
8,Blastoise,water,Misty
9,Caterpie,bug,Other Gym Leader


### Closing  the database connection

It is good practise to close the database connection when you are finsihed with a database.

This is done by calling the close() function with the connection.

In [10]:
# Closing the connection
data_conn.close()